# Run the public MaSIF-neosurface benchmark

Configure the published processed-data benchmark when you need the full application workflow.

This notebook previews the configuration by default. Edit your paths and set `RUN_TASK = True` when you are ready to run.


## 1. Before you start

- Install the `design` extra and obtain `weights/neosurface/ipa_12A_surface_atom.pt`.
- Obtain the upstream MaSIF-neosurf `computational_benchmark/benchmark_pdbs.txt` file.
- Prepare the published dataset with its `with_ligand`, `without_ligand`, and `decoys` directories.
- The archive is approximately 10 GB and extracted data require additional disk space. A full run is lengthy; use [seed docking](seed.ipynb) for a small first example.
- The full workflow may fetch original PDB structures. This notebook never starts downloads or the benchmark unless you enable the corresponding flag.


Use the `Python (masif_torch)` kernel. See [installation](../README.md#installation) and the [user guide](../USAGE.md).


## 2. Open the project


In [ ]:
from pathlib import Path
import importlib.util
import os
import shlex
import shutil
import subprocess
import sys
import tempfile

from omegaconf import OmegaConf

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "pyproject.toml").is_file() and (path / "examples").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the MaSIF_torch repository or its tutorials directory.")
os.chdir(ROOT)
CLI = [sys.executable, "-m", "masif"]
RUN_TASK = False  # Change to True only after editing the paths below.
print("Repository:", ROOT)
print("Python:", sys.executable)


## Optional: download and extract the published dataset

If the dataset is already installed, skip this section. Enable the download only
when you have enough disk space; extraction also takes time. An existing extracted
dataset should be used directly rather than extracted again.


In [ ]:
DOWNLOAD_DATASET = False
EXTRACT_DATASET = False
archive_directory = Path("data/benchmark")
dataset_directory = archive_directory / "dataset"

if DOWNLOAD_DATASET:
    subprocess.run([sys.executable, "tools/download_benchmark.py", "--output", str(archive_directory)], check=True)
if EXTRACT_DATASET:
    archive = archive_directory / "dataset.zip"
    if not archive.is_file():
        raise FileNotFoundError("Download the benchmark archive before enabling EXTRACT_DATASET.")
    subprocess.run([sys.executable, "tools/extract_benchmark.py", "--archive", str(archive),
                    "--output", str(dataset_directory)], check=True)
if not DOWNLOAD_DATASET and not EXTRACT_DATASET:
    print("Download and extraction are off. Point the configuration to an existing dataset.")


## 3. Choose your inputs and settings

Paths below are relative to the repository unless you provide an absolute path. Choose a separate output directory for each run.


In [ ]:
TASK = "benchmark"
config = OmegaConf.create({
    "task": "benchmark", "dataset": "data/benchmark/dataset",
    "definition": "data/upstream/masif-neosurf/computational_benchmark/benchmark_pdbs.txt",
    "checkpoint": "weights/neosurface/ipa_12A_surface_atom.pt",
    "output": "results/tutorial_benchmark", "workers": 1, "resume": True,
})
# Use the same inputs, checkpoint, and settings when resuming an interrupted run.


## 4. Preview your configuration

This prints the Hydra configuration without loading your input data or running the task.


In [ ]:
config_directory = tempfile.TemporaryDirectory(prefix="masif_tutorial_")
config_path = Path(config_directory.name) / "task.yaml"
OmegaConf.save(config, config_path)
command = CLI + [TASK, "--config", str(config_path)]
print(shlex.join(command))
subprocess.run(command + ["--cfg", "job"], check=True)


## 5. Run when ready

After editing the settings, set `RUN_TASK = True` in the setup cell and rerun the notebook. The checks below help identify missing inputs before starting.


In [ ]:
if RUN_TASK:
    for filename in (config.definition, config.checkpoint):
        if not Path(filename).is_file():
            raise FileNotFoundError(f"Missing {filename}. Obtain the definitions and checkpoint first.")
    for condition in ("with_ligand", "without_ligand", "decoys"):
        if not Path(config.dataset, condition).is_dir():
            raise FileNotFoundError(f"Missing dataset folder {condition}. Extract the full published dataset first.")
    if importlib.util.find_spec("open3d") is None:
        raise ImportError("Install the design extra from the README before running the benchmark.")
    subprocess.run(command, check=True)
else:
    print("Preview only. Edit your paths and set RUN_TASK = True to run.")


## 6. Next steps

Use the selected output directory for your run. To continue later, keep `resume: true` with the same inputs and settings. For everyday binder searches, use [seed docking](seed.ipynb) or [MaSIF-neosurface](neosurface.ipynb).
